# GMWI2 — The Gut Microbiome Wellness Index 2

> **Same note as notebook 07:** this notebook is conceptual, not runnable.
> GMWI2's real pipeline depends on MetaPhlAn3, Bowtie2, and Trimmomatic —
> compiled, conda-distributed tools that can't run in a browser sandbox.
> Treat this as a case study in how a research tool goes from a published
> paper to something a lab can actually run on a sample.

## The study

In 2024, a team including Mayo Clinic's Gut Microbiome and Mucosal Immunology
Laboratory (the Sung Lab / Kashyap Lab collaboration) published a tool called
the **Gut Microbiome Wellness Index 2 (GMWI2)** — a single number, computed
from a stool sample, that predicts whether someone's gut microbiome looks
more like a "healthy" or "non-healthy" profile.

What made it notable wasn't just the score — it was the scale behind it: the
model was trained on **8,069 stool shotgun metagenomes pooled from 54
published studies across 26 countries and six continents**, covering dozens
of different diseases. Evaluated by cross-validation, GMWI2 distinguishes
healthy from non-healthy individuals with roughly **80% balanced accuracy**,
rising above 90% for its highest-confidence predictions.

**Citation:** Chang, Gupta *et al.* (2024). *Gut Microbiome Wellness Index 2
enhances health status prediction from gut microbiome taxonomic profiles.*
Nature Communications. [doi:10.1038/s41467-024-51651-9](https://doi.org/10.1038/s41467-024-51651-9)
Code and data: [github.com/danielchang2002/GMWI2](https://github.com/danielchang2002/GMWI2)

## What GMWI2 is *not*

Worth stating plainly, in the same skeptical spirit as notebooks 03–04: GMWI2
does **not** diagnose any specific disease. It's trained across dozens of
unrelated conditions at once, so a low score means "this profile resembles
the disease-associated samples in the training set," not "you have condition
X." The authors position it as a general wellness signal — something that
might flag a worth-investigating change before symptoms show up, not a
replacement for an actual clinical diagnosis.

## The real pipeline

GMWI2 ships as a command-line tool (`gmwi2`, installable via `bioconda`).
Given two raw FASTQ files from a paired-end stool metagenome, it runs four
stages:

1. **Quality control** — strip adapter contamination (FastQC), remove human
   DNA reads that map to the human genome (Bowtie2 against GRCh38), trim
   low-quality bases (Trimmomatic)
2. **Taxonomic profiling** — run **MetaPhlAn3** against a marker-gene
   database to identify which microbial species are present and at what
   abundance
3. **Binarization** — convert species-level relative abundances into a
   simple **presence/absence** table (this species is here, or it isn't —
   the exact abundance stops mattering)
4. **Scoring** — feed the presence/absence vector into a pre-trained
   **Lasso-penalized logistic regression** model, which outputs the GMWI2
   score

Real usage looks like this:

```bash
# one-time setup
mamba install -c bioconda -c conda-forge gmwi2=1.6

# run on a sample's paired-end reads
gmwi2 -f forward.fastq -r reverse.fastq -n 8 -o output_prefix
```

```
output_prefix_GMWI2.txt          <- the score itself
output_prefix_GMWI2_taxa.txt     <- which taxa in this sample fed the score
output_prefix_metaphlan.txt      <- the raw MetaPhlAn3 taxonomic profile
```

A positive score leans "healthy"; a negative score leans "non-healthy." The
model is a Lasso logistic regression, which is worth noticing: it's the
*same family* of statistical model you'd reach for in a stats course, just
trained on a very large, very carefully pooled dataset — not some opaque
black box.

## The lighter path: scoring from an existing MetaPhlAn profile

If a lab has already run MetaPhlAn3 on its samples (a common situation —
plenty of pipelines produce a MetaPhlAn profile as a byproduct), GMWI2
provides a shortcut that skips straight to scoring, using the same
pre-trained model:

```bash
wget https://raw.githubusercontent.com/danielchang2002/GMWI2/main/src/gmwi2_metaphlan_output.py
wget https://raw.githubusercontent.com/danielchang2002/GMWI2/main/src/GMWI2/GMWI2_databases/GMWI2_model.joblib

python3 gmwi2_metaphlan_output.py metagenome_metaphlan_output.txt GMWI2_model.joblib output_prefix
```

This is just a Python script loading a `scikit-learn` model with `joblib`
and applying it to a table — no FASTQ, no Bowtie2, no compiled genomics
tools. It's the one piece of this notebook that's architecturally *close*
to something browser-runnable... which raises a natural question.

## Why you can't just point GMWI2 at this course's CSVs

Both `example_gut_samples.csv` and `real_gut_usa_malawi.csv` already exist
as finished abundance tables — so why not run the "lighter path" script
above directly on them? Two real, specific mismatches, not just red tape:

1. **Sequencing method.** This course's data comes from **16S rRNA
   sequencing** (notebook 01, section 7) — one marker gene, cheap, genus-level
   resolution. GMWI2 was trained on **shotgun metagenomic** data via
   MetaPhlAn3 — every gene, species-level (often strain-level) resolution.
   These aren't two file formats for the same measurement; they're two
   different assays with different statistical properties.
2. **Representation.** GMWI2's model was trained on **species-level
   presence/absence** (binary: detected or not), not genus-level percentage
   abundance. Even if you had shotgun data, you'd need to run it through
   MetaPhlAn3 first to get species names the trained model actually
   recognizes — the model's coefficients are tied to specific species IDs
   in its training data.

This is the same lesson as notebook 01's compositional-data caveat, one
level up: a number that looks like it *should* slot into a formula often
can't, because of what produced it. Checking that is part of the job.

### EXPLAIN #1

*GMWI2 collapses species-level abundance into presence/absence (just "is it
there," not "how much") before scoring. Given what you learned about
relative abundance and compositionality in notebook 01, why might throwing
away the exact abundance number be a deliberate design choice rather than a
loss of information?*

> your answer here

## Done — from a paper to a command line

You've now seen the same underlying shape twice: notebook 07's QIIME2/DADA2
pipeline turns raw reads into an abundance table; GMWI2 takes that same kind
of pipeline one step further, turning a taxonomic profile into a single
interpretable number, trained at a scale (8,069 samples, 54 studies) no
individual lab could collect alone. That combination — a large pooled
public dataset, a simple interpretable model, and a real published
benchmark — is what "a bioinformatics tool" looks like in practice, whether
it's QIIME2, GMWI2, or the next one.

**That's the full course.** If you want to go further: the GMWI2 GitHub
repo links a [Colab notebook](https://colab.research.google.com/github/danielchang2002/GMWI2/blob/main/manuscript/GMWI2_manuscript.ipynb)
that reproduces the paper's actual analysis on the full pooled dataset.